# 🏦 DARCRAYS — AI-Powered Alternate Credit Scoring
### Pipeline: Raw Transactions → Feature Engineering → GMM Clustering → XGBoost → CIBIL Score (300–900)

| Step | What happens |
|------|--------------|
| 0 | Install & Import |
| 1 | Generate Raw Transaction Data (1L users, 12–24 months) |
| 2 | Feature Engineering (62 features from raw txns) |
| 3 | Create Train / Validation / Test splits |
| 4 | Train GMM → Behavioral Clusters |
| 5 | GMM Imputation on Validation + QA |
| 6 | Train XGBoost Credit Scoring Model |
| 7 | Evaluate + Bias Audit |
| 8 | Predict on Test Set |
| 9 | Real-Time New User Scoring |

## 📦 Step 0 — Install & Import

In [ ]:
import subprocess, sys
pkgs = ['xgboost','shap','scikit-learn','pandas','numpy','matplotlib','seaborn','joblib']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', p, '-q'], check=False)
print('✅ packages ready')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, joblib, os, time, calendar
warnings.filterwarnings('ignore')

from numpy.random import default_rng
from datetime import datetime
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, confusion_matrix, roc_auc_score)
from sklearn.decomposition import PCA
import xgboost as xgb

os.makedirs('data',    exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('outputs', exist_ok=True)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print(f'✅ imports done | XGBoost {xgb.__version__}')

## 🏗️ Step 1 — Generate Raw Transaction Data
Generates **realistic raw bank statement transactions** for 1 lakh users covering **12–24 months** of banking history.  
Each user gets 80–200 transactions/month across 24 categories: SALARY_CREDIT, NACH_DEBIT, BNPL_REPAYMENT, GST_PAYMENT etc.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────
N_USERS      = 100_000
MONTHS_MIN   = 12
MONTHS_MAX   = 24
BASE_YEAR    = 2023
CHUNK_SIZE   = 5_000

USER_TYPES   = ['salaried_private','salaried_govt','shopkeeper','businessman','self_employed']
TYPE_WEIGHTS = [0.35, 0.15, 0.20, 0.18, 0.12]
BANDS        = ['A','B','C','D']
BAND_WEIGHTS = {
    'salaried_private': [0.30, 0.35, 0.25, 0.10],
    'salaried_govt':    [0.45, 0.35, 0.15, 0.05],
    'shopkeeper':       [0.20, 0.30, 0.30, 0.20],
    'businessman':      [0.22, 0.28, 0.28, 0.22],
    'self_employed':    [0.25, 0.32, 0.27, 0.16],
}
STRUCTURAL_ZERO_COLS = [
    'monthly_avg_business_credit','pos_txn_count_monthly_avg',
    'gst_payment_count','business_account_avg_balance',
    'trade_credit_utilisation','receivables_turnover_days',
]
META_COLS   = ['user_id','user_type']
TARGET_COLS = ['credit_score','risk_band']

CATEGORIES_DIR = {
    'SALARY_CREDIT':'CR','BUSINESS_CREDIT':'CR','POS_CREDIT':'CR',
    'UPI_CREDIT':'CR','CHEQUE_CREDIT':'CR','TRANSFER_CREDIT':'CR',
    'INTEREST_CREDIT':'CR','REFUND':'CR',
    'NACH_DEBIT':'DR','CHEQUE_BOUNCE_CHARGE':'DR','ELECTRICITY_BILL':'DR',
    'MOBILE_BILL':'DR','BROADBAND_BILL':'DR','INSURANCE_PREMIUM':'DR',
    'UPI_DEBIT':'DR','ATM_WITHDRAWAL':'DR','GROCERY_DEBIT':'DR',
    'ENTERTAINMENT_DEBIT':'DR','ONLINE_SHOPPING':'DR','BNPL_REPAYMENT':'DR',
    'RD_FD_INSTALLMENT':'DR','GST_PAYMENT':'DR','TRANSFER_DEBIT':'DR',
}
print('✅ Config ready')

In [ ]:
# ── HELPERS ─────────────────────────────────────────────────────
def _date(year, month, day):
    month = ((month-1)%12)+1
    year  = year + (month-1)//12
    _, mx = calendar.monthrange(year, month)
    return f'{year}-{month:02d}-{min(day,mx):02d}'

def _resolve_month(offset):
    yr = BASE_YEAR + (offset-1)//12
    mn = ((offset-1)%12)+1
    return yr, mn

def build_profile(uid, utype, band, rng):
    bp = {'A':{'qhi':(0.78,1.00),'qlo':(0.00,0.18)},
          'B':{'qhi':(0.55,0.82),'qlo':(0.12,0.40)},
          'C':{'qhi':(0.28,0.58),'qlo':(0.35,0.68)},
          'D':{'qhi':(0.00,0.32),'qlo':(0.60,1.00)}}[band]
    qhi = float(rng.uniform(*bp['qhi']))
    qlo = float(rng.uniform(*bp['qlo']))
    inc_range = {'salaried_private':(18000,200000),'salaried_govt':(25000,180000),
                 'shopkeeper':(15000,120000),'businessman':(30000,800000),
                 'self_employed':(20000,300000)}[utype]
    inc    = float(rng.uniform(*inc_range))
    n_mon  = int(rng.integers(MONTHS_MIN, MONTHS_MAX+1))
    sal_d  = int(rng.integers(1,10))
    jitter = 0 if band in ['A','B'] else int(rng.integers(0,6))
    has_emi = bool(rng.random() > (0.1 if band=='D' else 0.3))
    emi_amt = inc*float(rng.uniform(0.08,0.45)) if has_emi else 0.0
    emi_cnt = int(rng.integers(1,5)) if has_emi else 0
    is_biz  = utype in ['shopkeeper','businessman']
    sales   = inc/float(rng.uniform(0.12,0.45)) if is_biz else 0.0
    return {'user_id':uid,'user_type':utype,'true_band':band,
            'age':int(rng.integers(22,65)),'n_months':n_mon,
            'monthly_income':inc,'sal_day':sal_d,'sal_jitter':jitter,
            'has_emi':has_emi,'emi_amount':emi_amt,'emi_count':emi_cnt,
            'emi_ontime_prob':0.45+qhi*0.55,'is_business':is_biz,
            'monthly_sales':sales,'employer_tenure':int(rng.integers(1,20)),
            'account_vintage':int(rng.integers(6,240)),
            'kyc_score':float(0.5+qhi*0.5),'co_applicant':int(rng.integers(0,2)),
            'qhi':qhi,'qlo':qlo}

print('✅ helpers defined')

In [ ]:
def generate_user_transactions(p, rng):
    uid,utype,inc,qhi,qlo = p['user_id'],p['user_type'],p['monthly_income'],p['qhi'],p['qlo']
    rows=[]
    def add(yr,mn,day,cat,amt):
        rows.append({'user_id':uid,'date':_date(yr,mn,day),'month':mn,
                     'category':cat,'direction':CATEGORIES_DIR[cat],'amount':round(max(float(amt),1.0),2)})

    util_p  = 0.35+qhi*0.65
    bnpl_p  = 0.05+qlo*0.40

    for mo in range(1, p['n_months']+1):
        yr,mn = _resolve_month(mo)
        _,mx  = calendar.monthrange(yr,mn)
        def rd(lo=1,hi=None): return int(rng.integers(lo,(hi or mx)+1))

        # Income credits
        if utype in ['salaried_private','salaried_govt']:
            j = int(rng.integers(-p['sal_jitter'],p['sal_jitter']+1)) if p['sal_jitter']>0 else 0
            sd= max(1,min(mx,p['sal_day']+j))
            if utype=='salaried_govt' or rng.random()>0.08*(1+qlo):
                add(yr,mn,sd,'SALARY_CREDIT',inc*float(rng.uniform(0.97,1.03)))
        elif p['is_business']:
            for _ in range(int(rng.integers(15,60))):
                add(yr,mn,rd(),'POS_CREDIT',p['monthly_sales']/35*float(rng.uniform(0.4,1.8)))
            if rng.random()>0.45: add(yr,mn,rd(),'BUSINESS_CREDIT',p['monthly_sales']*float(rng.uniform(0.05,0.25)))
        else:
            for _ in range(int(rng.integers(3,12))): add(yr,mn,rd(),'UPI_CREDIT',inc/7*float(rng.uniform(0.6,1.5)))
            if rng.random()>0.4: add(yr,mn,rd(),'CHEQUE_CREDIT',inc*float(rng.uniform(0.15,0.5)))

        for _ in range(int(rng.integers(0,6))): add(yr,mn,rd(),'UPI_CREDIT',float(rng.uniform(100,4000)))
        if mn%3==0: add(yr,mn,mx,'INTEREST_CREDIT',float(rng.uniform(50,2500)))

        # EMI
        if p['has_emi'] and p['emi_count']>0:
            for _ in range(p['emi_count']):
                ep = p['emi_amount']/p['emi_count']
                if rng.random()<p['emi_ontime_prob']:
                    add(yr,mn,min(mx,p['sal_day']+3),'NACH_DEBIT',ep)
                else:
                    add(yr,mn,rd(),'CHEQUE_BOUNCE_CHARGE',ep*float(rng.uniform(1.01,1.05)))
                    if rng.random()>0.35: add(yr,mn,min(mx,rd()+3),'NACH_DEBIT',ep)

        # Utility bills
        for cat,frac in [('ELECTRICITY_BILL',0.04),('MOBILE_BILL',0.018),('BROADBAND_BILL',0.012)]:
            if rng.random()<util_p: add(yr,mn,rd(1,10),cat,inc*frac*float(rng.uniform(0.80,1.25)))
        if mn%3==1 and rng.random()<(0.3+qhi*0.5): add(yr,mn,rd(),'INSURANCE_PREMIUM',inc*float(rng.uniform(0.02,0.08)))

        # Spending
        gb = inc*float(rng.uniform(0.07,0.20))
        for _ in range(int(rng.integers(4,14))): add(yr,mn,rd(),'GROCERY_DEBIT',gb/9*float(rng.uniform(0.5,1.6)))
        for _ in range(int(rng.integers(0,5))): add(yr,mn,rd(),'ATM_WITHDRAWAL',float(rng.choice([500,1000,2000,3000,5000])))
        sb = inc*(0.30+qlo*0.50)
        for _ in range(int(rng.integers(8,45))): add(yr,mn,rd(),'UPI_DEBIT',sb/25*float(rng.uniform(0.3,2.2)))
        if qlo>0.05:
            for _ in range(int(rng.integers(0,max(1,int(qlo*8))))): add(yr,mn,rd(),'ENTERTAINMENT_DEBIT',inc*qlo*0.04*float(rng.uniform(0.5,2.0)))
            for _ in range(int(rng.integers(0,max(1,int(qlo*5))))): add(yr,mn,rd(),'ONLINE_SHOPPING',inc*qlo*0.05*float(rng.uniform(0.5,2.5)))
        if rng.random()<bnpl_p: add(yr,mn,rd(),'BNPL_REPAYMENT',inc*float(rng.uniform(0.01,0.10)))
        if rng.random()<(0.10+qhi*0.55): add(yr,mn,rd(1,5),'RD_FD_INSTALLMENT',inc*float(rng.uniform(0.03,0.18)))
        if p['is_business'] and rng.random()<(0.3+qhi*0.6): add(yr,mn,rd(10,20),'GST_PAYMENT',p['monthly_sales']*float(rng.uniform(0.03,0.12)))

    return rows

print('✅ transaction generator defined')

In [ ]:
print('='*60)
print(f'Generating raw transactions for {N_USERS:,} users ...')
print('='*60)

rng_main   = default_rng(42)
type_idx   = rng_main.choice(len(USER_TYPES), size=N_USERS, p=TYPE_WEIGHTS)
assignments= []
for ti in type_idx:
    ut   = USER_TYPES[ti]
    band = str(rng_main.choice(BANDS, p=BAND_WEIGHTS[ut]))
    assignments.append((ut,band))

all_profiles = []
txn_buffer   = []
first_write  = True
total_txns   = 0
t0 = time.time()

for i,(utype,band) in enumerate(assignments):
    seed    = hash((42,i)) & 0xFFFFFFFF
    rng     = default_rng(seed)
    profile = build_profile(i,utype,band,rng)
    txns    = generate_user_transactions(profile,rng)

    all_profiles.append({'user_id':profile['user_id'],'user_type':profile['user_type'],
        'true_band':profile['true_band'],'age':profile['age'],
        'n_months':profile['n_months'],'monthly_income':round(profile['monthly_income'],2),
        'has_emi':int(profile['has_emi']),'is_business':int(profile['is_business']),
        'employer_tenure':profile['employer_tenure'],'account_vintage':profile['account_vintage'],
        'kyc_score':round(profile['kyc_score'],4),'co_applicant':profile['co_applicant'],
        'qhi':round(profile['qhi'],4),'qlo':round(profile['qlo'],4)})

    txn_buffer.extend(txns)
    total_txns += len(txns)

    if (i+1)%CHUNK_SIZE==0 or (i+1)==N_USERS:
        pd.DataFrame(txn_buffer).to_csv('data/raw_transactions.csv',
            mode='w' if first_write else 'a', header=first_write, index=False)
        first_write = False
        txn_buffer  = []
        elapsed = time.time()-t0
        print(f'  [{i+1:>7,}/{N_USERS:,}]  txns={total_txns:>10,}  elapsed={elapsed:.0f}s')

df_profiles = pd.DataFrame(all_profiles)
df_profiles.to_csv('data/user_profiles.csv', index=False)

print(f'\n✅ Done in {time.time()-t0:.0f}s')
print(f'   data/raw_transactions.csv  — {total_txns:,} rows')
print(f'   data/user_profiles.csv     — {N_USERS:,} users')

In [ ]:
# Peek at raw data
df_txn_sample = pd.read_csv('data/raw_transactions.csv', nrows=5000)
print('Sample transactions:')
display(df_txn_sample.head(10))
print(f'\nCategory distribution (sample 5k rows):')
display(df_txn_sample['category'].value_counts())
print(f'\nAvg txns per user (full): {total_txns/N_USERS:.0f}')

## 🔧 Step 2 — Feature Engineering
Reads raw transactions → computes **62 behavioral features** per user.  
All ratios, averages, counts are computed HERE from raw data — not pre-baked.

In [ ]:
SCORE_WEIGHTS = {
    'income':0.20,'balance':0.15,'spending':0.15,'emi':0.20,
    'bills':0.15,'savings':0.07,'digital':0.04,'business':0.05,
    'relation':0.03,'bnpl':0.02,
}
WEIGHT_NORM = sum(SCORE_WEIGHTS.values())  # 1.06
print(f'Score weight sum = {WEIGHT_NORM}  (will normalise)')

In [ ]:
def engineer_features(uid, txns, profile):
    if txns.empty: return None
    utype = profile['user_type']
    n_mon = int(profile['n_months'])

    def msum(cat):
        s = txns[txns['category']==cat].groupby('month')['amount'].sum()
        return s.reindex(range(1,n_mon+1), fill_value=0.0)
    def mcount(cat):
        s = txns[txns['category']==cat].groupby('month')['amount'].count()
        return s.reindex(range(1,n_mon+1), fill_value=0)

    cr = txns[txns['direction']=='CR'].groupby('month')['amount'].sum().reindex(range(1,n_mon+1),fill_value=0.0)
    dr = txns[txns['direction']=='DR'].groupby('month')['amount'].sum().reindex(range(1,n_mon+1),fill_value=0.0)

    sal=msum('SALARY_CREDIT'); pos=msum('POS_CREDIT'); biz=msum('BUSINESS_CREDIT')
    upi_c=msum('UPI_CREDIT');  chq_c=msum('CHEQUE_CREDIT')
    nach=msum('NACH_DEBIT');   bounce=mcount('CHEQUE_BOUNCE_CHARGE')
    elec=msum('ELECTRICITY_BILL'); mob=msum('MOBILE_BILL'); bb=msum('BROADBAND_BILL')
    ins=msum('INSURANCE_PREMIUM'); groc=msum('GROCERY_DEBIT')
    atm=msum('ATM_WITHDRAWAL');    upi_d=msum('UPI_DEBIT')
    ent=msum('ENTERTAINMENT_DEBIT'); shop=msum('ONLINE_SHOPPING')
    bnpl=msum('BNPL_REPAYMENT');   rdfd=msum('RD_FD_INSTALLMENT')
    gst=msum('GST_PAYMENT')

    tcr = float(cr.sum())+1e-6
    tdr = float(dr.sum())+1e-6

    main_inc = sal if utype in ['salaried_private','salaried_govt'] \
               else (pos+biz if profile['is_business'] else upi_c+chq_c)
    avg_inc  = float(main_inc.mean())+1e-6

    # Salary day consistency
    stxn = txns[txns['category']=='SALARY_CREDIT'].copy()
    if not stxn.empty:
        stxn['day'] = pd.to_datetime(stxn['date']).dt.day
        std_d = float(stxn['day'].std()) if len(stxn)>1 else 0.0
        sal_cons = float(np.clip(1.0-std_d/10.0,0,1))
    else: sal_cons=0.0

    h1 = float(main_inc.iloc[:n_mon//2].mean())+1e-6
    h2 = float(main_inc.iloc[n_mon//2:].mean())+1e-6
    inc_growth = float(np.clip((h2-h1)/h1,-0.5,1.0))
    inc_cv     = float(main_inc.std()/avg_inc)
    sec_ratio  = float(np.clip(upi_c.sum()/tcr,0,0.6)) if utype in ['salaried_private','salaried_govt'] else 0.0
    io_ratio   = float(np.clip(tcr/tdr,0.5,4.0))

    # Balance proxy
    bal = (cr-dr).cumsum()
    avg_bal  = float(bal.mean())
    bal_vol  = float(bal.std())
    neg_mons = int((bal<0).sum())
    low_mons = int((bal<5000).sum())
    bal_util = float(np.clip(tdr/tcr,0.05,1.5))

    # Spending
    groc_r = float(np.clip(groc.sum()/tcr,0,0.5))
    util_r = float(np.clip((elec+mob+bb).sum()/tcr,0,0.35))
    ent_r  = float(np.clip(ent.sum()/tcr,0,0.35))
    atm_r  = float(np.clip(atm.sum()/tcr,0,0.6))
    shop_r = float(np.clip(shop.sum()/tcr,0,0.35))
    spd_r  = float(np.clip(tdr/tcr,0.2,1.5))
    upi_m  = float(mcount('UPI_DEBIT').mean())

    # EMI
    avg_nach  = float(nach.mean())
    emi_ratio = float(np.clip(avg_nach/avg_inc,0,0.8))
    b_total   = int(bounce.sum())
    nach_att  = int(txns[txns['category'].isin(['NACH_DEBIT','CHEQUE_BOUNCE_CHARGE'])].shape[0])
    nach_ok   = int(txns[txns['category']=='NACH_DEBIT'].shape[0])
    si_rate   = float(np.clip(nach_ok/(nach_att+1e-6),0,1))
    has_emi   = bool(profile['has_emi'])
    loan_trk  = float(np.clip(0.3+float(profile['kyc_score'])*0.7,0,1)) if has_emi \
                else float(np.clip(0.7+float(profile['kyc_score'])*0.3,0,1))

    # Bills
    elec_ok=int((elec>0).sum()); mob_ok=int((mob>0).sum())
    bb_ok=int((bb>0).sum());     ins_ok=int((ins>0).sum())
    util_cons=float(np.clip((elec_ok+mob_ok+bb_ok)/(n_mon*3),0,1))

    # Savings
    net_sav  = float(np.clip((tcr-tdr)/tcr,-0.3,0.7))
    rdfd_r   = float(np.clip(rdfd.sum()/tcr,0,0.45))
    rdfd_cnt = int(np.clip((rdfd>0).sum()//4,0,5))
    sav_txn  = int(np.clip((rdfd>0).sum()*2,0,24))

    # Digital
    qhi_proxy= max(float(profile['kyc_score'])-0.5,0)
    nb_days  = float(np.clip(5+qhi_proxy*40,0,25))
    app_sess = float(np.clip(3+qhi_proxy*114,0,60))
    upi_auto = int(nach_ok>0)
    dc_txn   = float(np.clip(txns[txns['category'].isin(['GROCERY_DEBIT','ENTERTAINMENT_DEBIT','ONLINE_SHOPPING'])].shape[0],0,200))

    # BNPL
    bnpl_mon = int((bnpl>0).sum())
    bnpl_tot = float(bnpl.sum())
    bnpl_ir  = float(np.clip(bnpl_tot/tcr,0,0.5))
    bnpl_ot  = float(np.clip(1.0-bnpl_ir*3,0,1))

    # Business
    is_biz = bool(profile['is_business'])
    if is_biz:
        avg_biz = float((pos+biz).mean())
        gst_cnt = int((gst>0).sum())
        trade_u = float(np.clip(0.1+(1-float(profile['kyc_score']))*0.7,0,0.9))
        recv_d  = float(np.clip(5+(1-float(profile['kyc_score']))*115,5,120))
        pos_m   = float(mcount('POS_CREDIT').mean())
        biz_bal = float(np.clip(avg_bal*1.5,0,5e6))
    else:
        avg_biz=pos_m=gst_cnt=trade_u=recv_d=biz_bal=0.0

    vintage   = int(profile['account_vintage'])
    rel_score = float(np.clip(float(profile['kyc_score'])*0.8+0.1,0,1))

    return {
        # Income
        'monthly_avg_income':float(avg_inc),'salary_credit_count':int(stxn.shape[0]) if utype in ['salaried_private','salaried_govt'] else int(txns[txns['direction']=='CR'].shape[0]),
        'income_variability_cv':float(np.clip(inc_cv,0,2)),'salary_day_consistency':float(sal_cons),
        'employer_tenure_years':float(profile['employer_tenure']),'income_growth_yoy':float(inc_growth),
        'secondary_income_ratio':float(sec_ratio),'total_annual_inflow':float(cr.sum()),
        'inflow_outflow_ratio':float(io_ratio),
        # Balance
        'avg_monthly_balance':float(max(avg_bal,0)),'min_balance_breach_count':int(np.clip(low_mons*1.5,0,24)),
        'balance_below_5k_months':int(np.clip(low_mons,0,n_mon)),'avg_eom_balance':float(max(avg_bal,0)),
        'balance_volatility':float(np.clip(bal_vol,0,1e7)),'negative_balance_months':int(np.clip(neg_mons,0,n_mon)),
        'balance_utilisation_ratio':float(np.clip(bal_util,0.05,1.5)),
        # Spending
        'monthly_avg_debit':float(dr.mean()),'debit_txn_count_monthly':float(txns[txns['direction']=='DR'].shape[0]/n_mon),
        'grocery_spend_ratio':float(groc_r),'utility_spend_ratio':float(util_r),
        'entertainment_spend_ratio':float(ent_r),'atm_withdrawal_ratio':float(atm_r),
        'upi_txn_count_monthly':float(upi_m),'online_shopping_ratio':float(shop_r),
        'spend_to_income_ratio':float(np.clip(spd_r,0.2,1.5)),'weekend_spend_ratio':float(np.clip(0.20+float(np.random.uniform(0,0.25)),0.1,0.55)),
        # EMI
        'active_emi_count':int(has_emi)*int(max(1,int(np.random.randint(1,5)))) if has_emi else 0,
        'monthly_emi_obligation':float(avg_nach),'emi_to_income_ratio':float(emi_ratio),
        'emi_bounce_count':int(np.clip(b_total,0,12)),'emi_paid_ontime_ratio':float(si_rate),
        'nach_mandate_active':int(has_emi),'loan_repayment_track_score':float(loan_trk),
        # Bills
        'electricity_paid_ontime_months':int(np.clip(elec_ok,0,n_mon)),
        'mobile_paid_ontime_months':int(np.clip(mob_ok,0,n_mon)),
        'broadband_paid_ontime_months':int(np.clip(bb_ok,0,n_mon)),
        'insurance_paid_months':int(np.clip(ins_ok,0,n_mon)),
        'utility_payment_consistency':float(util_cons),'cheque_bounce_count':int(np.clip(b_total,0,10)),
        'standing_instruction_rate':float(si_rate),
        # Savings
        'rd_fd_count_active':int(rdfd_cnt),'savings_txn_count':int(sav_txn),
        'net_savings_rate':float(np.clip(net_sav,-0.3,0.7)),'investment_inflow_ratio':float(rdfd_r),
        'sweep_utilisation':float(np.clip(float(np.random.uniform(0,1)),0,1)),
        # Digital
        'netbanking_days_monthly':float(nb_days),'app_sessions_monthly':float(app_sess),
        'upi_autopay_mandates':int(upi_auto),'debit_card_txn_count':float(dc_txn),
        # BNPL
        'bnpl_usage_months':int(np.clip(bnpl_mon,0,n_mon)),'bnpl_repayment_ratio':float(bnpl_ot),
        'bnpl_to_income_ratio':float(bnpl_ir),
        # Business (structural zeros for non-business)
        'monthly_avg_business_credit':float(avg_biz),'pos_txn_count_monthly_avg':float(pos_m),
        'gst_payment_count':int(gst_cnt),'business_account_avg_balance':float(biz_bal),
        'trade_credit_utilisation':float(trade_u),'receivables_turnover_days':float(recv_d),
        # Profile
        'age':int(profile['age']),'account_vintage_months':int(np.clip(vintage,6,240)),
        'kyc_completeness_score':float(np.clip(float(profile['kyc_score']),0.5,1.0)),
        'co_applicant_flag':int(profile['co_applicant']),'existing_relationship_score':float(rel_score),
        'history_months':int(n_mon),
    }

print('✅ engineer_features() defined')

In [ ]:
def compute_score(f):
    def c(x): return float(np.clip(x,0.0,1.0))
    h = int(f['history_months'])+1e-6
    inc = c(0.25*c(f['salary_day_consistency'])+0.20*c(f['income_growth_yoy']/0.40+0.375)+
            0.20*c(1-f['income_variability_cv']/1.0)+0.20*c((f['inflow_outflow_ratio']-0.85)/1.65)+
            0.15*c(f['net_savings_rate']/0.55+0.18))
    bal = c(0.30*c(1-f['balance_below_5k_months']/h)+0.25*c(1-f['negative_balance_months']/h)+
            0.25*c(1-min(f['balance_utilisation_ratio'],1.0))+0.20*c(1-f['min_balance_breach_count']/24))
    spd = c(0.35*c(1-f['spend_to_income_ratio'])+0.30*c(1-f['entertainment_spend_ratio']/0.25)+
            0.20*c(1-f['online_shopping_ratio']/0.30)+0.15*c(f['net_savings_rate']/0.55+0.18))
    emi = c(0.35*c(f['emi_paid_ontime_ratio'])+0.30*c(f['loan_repayment_track_score'])+
            0.20*c(1-f['emi_bounce_count']/12)+0.15*c(1-min(f['emi_to_income_ratio']/0.55,1.0)))
    bill= c(0.22*c(f['electricity_paid_ontime_months']/h)+0.18*c(f['mobile_paid_ontime_months']/h)+
            0.22*c(f['utility_payment_consistency'])+0.22*c(f['standing_instruction_rate'])+
            0.16*c(1-f['cheque_bounce_count']/10))
    sav = c(0.40*c(f['net_savings_rate']/0.55+0.18)+0.35*c(f['investment_inflow_ratio']/0.40)+0.25*c(f['rd_fd_count_active']/5))
    dig = c(0.50*c(f['app_sessions_monthly']/60)+0.30*c(f['netbanking_days_monthly']/25)+0.20*c(f['upi_autopay_mandates']/8))
    biz = c(0.35*c(f['gst_payment_count']/h)+0.35*c(1-f['trade_credit_utilisation'])+
            0.30*c(1-f['receivables_turnover_days']/120)) if f['monthly_avg_business_credit']>0 else 0.5
    rel = c(0.50*c(f['account_vintage_months']/240)+0.30*c(f['existing_relationship_score'])+0.20*c(f['kyc_completeness_score']))
    bnp = c(0.60*c(f['bnpl_repayment_ratio'])+0.40*c(1-f['bnpl_to_income_ratio']/0.3))
    comp= (SCORE_WEIGHTS['income']*inc+SCORE_WEIGHTS['balance']*bal+SCORE_WEIGHTS['spending']*spd+
           SCORE_WEIGHTS['emi']*emi+SCORE_WEIGHTS['bills']*bill+SCORE_WEIGHTS['savings']*sav+
           SCORE_WEIGHTS['digital']*dig+SCORE_WEIGHTS['business']*biz+
           SCORE_WEIGHTS['relation']*rel+SCORE_WEIGHTS['bnpl']*bnp)/WEIGHT_NORM
    return int(np.clip(300+comp*600,300,900))

def assign_band(s):
    return 'A' if s>=750 else ('B' if s>=650 else ('C' if s>=550 else 'D'))

def to_band(scores):
    scores=np.asarray(scores)
    return np.where(scores>=750,'A',np.where(scores>=650,'B',np.where(scores>=550,'C','D')))

print('✅ scoring functions defined')

In [ ]:
print('Loading raw transactions ...')
t0 = time.time()
df_txn  = pd.read_csv('data/raw_transactions.csv', dtype={'user_id':int,'month':int,'amount':float})
prof_df = pd.read_csv('data/user_profiles.csv')
profiles_dict = prof_df.set_index('user_id').to_dict('index')
txn_grp = df_txn.groupby('user_id')
print(f'Loaded {len(df_txn):,} transactions in {time.time()-t0:.1f}s')

print('\nEngineering features ...')
t0   = time.time()
rows = []
for i, uid in enumerate(profiles_dict.keys()):
    if uid not in txn_grp.groups: continue
    feat = engineer_features(uid, txn_grp.get_group(uid), profiles_dict[uid])
    if feat is None: continue
    feat['user_id']      = uid
    feat['user_type']    = profiles_dict[uid]['user_type']
    feat['credit_score'] = compute_score(feat)
    feat['risk_band']    = assign_band(feat['credit_score'])
    rows.append(feat)
    if (i+1)%10000==0: print(f'  {i+1:>7,}/{len(profiles_dict):,}  ({time.time()-t0:.0f}s)')

df_all = pd.DataFrame(rows)
feat_cols = [c for c in df_all.columns if c not in META_COLS+TARGET_COLS]
df_all    = df_all[feat_cols+META_COLS+TARGET_COLS]

print(f'\n✅ Features engineered in {time.time()-t0:.0f}s')
print(f'Shape       : {df_all.shape}')
print(f'Score range : {df_all["credit_score"].min()} – {df_all["credit_score"].max()}')
print(f'Mean score  : {df_all["credit_score"].mean():.0f}')
print(f'Bands:\n{df_all["risk_band"].value_counts().sort_index().to_string()}')

## 📊 Step 3 — EDA + Create Train / Validation / Test Splits

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))

axes[0].hist(df_all['credit_score'],bins=50,color='steelblue',edgecolor='white',lw=0.4)
axes[0].axvline(df_all['credit_score'].mean(),color='red',ls='--',label=f'Mean={df_all["credit_score"].mean():.0f}')
for thr,col in [(750,'#2ecc71'),(650,'#f39c12'),(550,'#e74c3c')]:
    axes[0].axvline(thr,color=col,ls=':',lw=1.5)
axes[0].set_title('Credit Score Distribution (300–900)',fontweight='bold')
axes[0].set_xlabel('Score'); axes[0].legend()

utypes=[]; data_bp=[]
for u in df_all['user_type'].unique():
    utypes.append(u); data_bp.append(df_all[df_all['user_type']==u]['credit_score'].values)
bp = axes[1].boxplot(data_bp,labels=utypes,patch_artist=True)
colors=plt.cm.Set2(np.linspace(0,1,len(utypes)))
for patch,c in zip(bp['boxes'],colors): patch.set_facecolor(c)
axes[1].set_title('Score by User Type',fontweight='bold'); axes[1].tick_params(axis='x',rotation=25)

bc=df_all['risk_band'].value_counts().sort_index()
bc_colors={'A':'#2ecc71','B':'#f39c12','C':'#e67e22','D':'#e74c3c'}
axes[2].bar(bc.index,bc.values,color=[bc_colors[b] for b in bc.index],edgecolor='white')
for i,(b,cnt) in enumerate(bc.items()):
    axes[2].text(i,cnt+500,f'{cnt:,}\n({cnt/len(df_all)*100:.1f}%)',ha='center',fontsize=8,fontweight='bold')
axes[2].set_title('Risk Band Distribution',fontweight='bold')

plt.suptitle('Feature-Engineered Dataset — EDA',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/eda_overview.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
# ── DETECT DATA LEAKAGE ──────────────────────────────────────────
corr = df_all[feat_cols+['credit_score']].select_dtypes(include=np.number).corr()['credit_score'].drop('credit_score')
leak_features = corr[corr.abs()>0.90].index.tolist()
print(f'Leak features (|corr|>0.90): {leak_features}')
feat_cols_clean = [c for c in feat_cols if c not in leak_features]
print(f'Clean feature count: {len(feat_cols_clean)}')

# ── TRAIN / VAL / TEST SPLIT (70 / 20 / 10) ─────────────────────
rng_sp = default_rng(99)
idx    = rng_sp.permutation(len(df_all))
n_tr   = int(len(df_all)*0.70)
n_vl   = int(len(df_all)*0.20)

df_train = df_all.iloc[idx[:n_tr]].reset_index(drop=True)
df_val   = df_all.iloc[idx[n_tr:n_tr+n_vl]].reset_index(drop=True)
df_test  = df_all.iloc[idx[n_tr+n_vl:]].reset_index(drop=True)

print(f'\nSplit sizes:')
print(f'  Training   : {len(df_train):,} users')
print(f'  Validation : {len(df_val):,}   users')
print(f'  Test       : {len(df_test):,}   users')

# ── INTRODUCE MISSINGNESS (val 30%, test 25%) ────────────────────
imputable = [c for c in feat_cols_clean if c not in STRUCTURAL_ZERO_COLS]

def apply_missing(df, ratio, seed):
    rng_m=default_rng(seed); df=df.copy()
    for i in range(len(df)):
        miss=rng_m.choice(imputable,size=max(1,int(len(imputable)*ratio)),replace=False)
        for col in miss: df.iloc[i,df.columns.get_loc(col)]=np.nan
    return df

df_val_gt   = df_val.copy()    # ground truth before missingness
df_test_gt  = df_test.copy()
df_val_miss = apply_missing(df_val, 0.30, seed=200)
df_test_miss= apply_missing(df_test,0.25, seed=300)
df_test_nolabel = df_test_miss.drop(columns=['credit_score','risk_band'])

# Save
df_train.to_csv('data/training.csv',index=False)
df_val_miss.to_csv('data/validation.csv',index=False)
df_test_nolabel.to_csv('data/test.csv',index=False)
df_val_gt.to_csv('data/validation_ground_truth.csv',index=False)
df_test_gt.to_csv('data/test_ground_truth.csv',index=False)

print(f'\n✅ Datasets saved:')
print(f'  training.csv         — {len(df_train):,} rows | 0% missing | with labels')
print(f'  validation.csv       — {len(df_val_miss):,} rows | ~30% missing | with labels')
print(f'  test.csv             — {len(df_test_nolabel):,} rows | ~25% missing | NO labels')
print(f'  Val missing avg      : {df_val_miss[feat_cols_clean].isnull().mean().mean()*100:.1f}%')

## 🔬 Step 4 — Train GMM (Behavioral Clustering)
GMM assigns **soft probabilistic membership** across clusters.  
A user can be 40% Cluster-2 (stable salaried) + 35% Cluster-5 (irregular businessman).

In [ ]:
X_tr = df_train[feat_cols_clean].copy()
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
print(f'Scaled: mean≈{X_tr_scaled.mean():.3f}  std≈{X_tr_scaled.std():.3f}')

In [ ]:
rng_sub = np.random.default_rng(42)
idx_sub = rng_sub.choice(len(X_tr_scaled), 10_000, replace=False)
X_sub   = X_tr_scaled[idx_sub]

bic_scores, aic_scores, n_range = [], [], range(4, 15)
for n in n_range:
    g = GaussianMixture(n_components=n,covariance_type='diag',random_state=42,max_iter=150,n_init=2)
    g.fit(X_sub)
    bic_scores.append(g.bic(X_sub))
    aic_scores.append(g.aic(X_sub))
    print(f'  n={n:2d} | BIC={bic_scores[-1]:>12.0f} | AIC={aic_scores[-1]:>12.0f}')

optimal_n = list(n_range)[int(np.argmin(bic_scores))]
print(f'\n✅ Optimal n_components = {optimal_n}')

fig,ax=plt.subplots(figsize=(9,4))
ax.plot(list(n_range),bic_scores,'o-',color='steelblue',lw=2,label='BIC')
ax.plot(list(n_range),aic_scores,'s--',color='tomato',lw=2,label='AIC')
ax.axvline(optimal_n,color='green',ls=':',lw=2,label=f'Optimal n={optimal_n}')
ax.set_xlabel('Number of Components'); ax.set_ylabel('Score (lower=better)')
ax.set_title('GMM Model Selection: BIC & AIC',fontweight='bold')
ax.legend(); ax.set_xticks(list(n_range))
plt.tight_layout(); plt.savefig('outputs/gmm_bic_aic.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
print(f'Training GMM (n={optimal_n}) on {len(X_tr_scaled):,} rows ...')
t0 = time.time()
gmm = GaussianMixture(n_components=optimal_n,covariance_type='diag',random_state=42,max_iter=400,n_init=3)
gmm.fit(X_tr_scaled)
print(f'✅ Converged={gmm.converged_}  log-likelihood={gmm.lower_bound_:.4f}  ({time.time()-t0:.0f}s)')

hard_labels = gmm.predict(X_tr_scaled)
df_train_c  = df_train.copy()
df_train_c['gmm_cluster'] = hard_labels

cprof = df_train_c.groupby('gmm_cluster')['credit_score'].agg(['mean','std','count'])
cprof.columns=['avg_score','std_score','count']
cprof['profile']=cprof['avg_score'].apply(lambda s:'LOW RISK' if s>=700 else('MEDIUM' if s>=580 else 'HIGH RISK'))
print('\nCluster Profiles:')
print(cprof.to_string())

# PCA visualization
pca=PCA(n_components=2,random_state=42)
idx5k=np.random.choice(len(X_tr_scaled),5000,replace=False)
X_pca=pca.fit_transform(X_tr_scaled[idx5k])

fig,axes=plt.subplots(1,2,figsize=(16,6))
sc1=axes[0].scatter(X_pca[:,0],X_pca[:,1],c=hard_labels[idx5k],cmap='tab10',alpha=0.4,s=6)
axes[0].set_title(f'GMM Clusters (n={optimal_n}) — PCA',fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(sc1,ax=axes[0])

sc2=axes[1].scatter(X_pca[:,0],X_pca[:,1],c=df_train['credit_score'].values[idx5k],
                    cmap='RdYlGn',alpha=0.4,s=6,vmin=300,vmax=900)
axes[1].set_title('Credit Score Gradient — PCA',fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(sc2,ax=axes[1],label='Credit Score')
plt.suptitle('Behavioural Cluster Structure',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/gmm_clusters_pca.png',dpi=150,bbox_inches='tight'); plt.show()

## 🔧 Step 5 — GMM Imputation on Validation + QA
Missing value = **weighted average of cluster means**, weights = soft cluster membership probabilities.

In [ ]:
def gmm_impute(df_in, gmm_model, scaler_obj, feat_cols):
    X         = df_in[feat_cols].copy().values.astype(float)
    mask      = np.isnan(X)
    has_miss  = mask.any(axis=1)
    if not has_miss.any(): return df_in.copy()

    mu_scaled = gmm_model.means_
    mu_orig   = scaler_obj.inverse_transform(gmm_model.means_)
    cov_diag  = gmm_model.covariances_
    log_w     = np.log(gmm_model.weights_+1e-10)
    K         = gmm_model.n_components

    for i in np.where(has_miss)[0]:
        miss_c = np.where(mask[i])[0]
        obs_c  = np.where(~mask[i])[0]
        if len(obs_c)==0:
            probs=np.exp(log_w-log_w.max()); probs/=probs.sum()
        else:
            tmp=X[i].copy(); tmp[miss_c]=mu_orig[:,miss_c].mean(axis=0)
            tmp_s=scaler_obj.transform(tmp.reshape(1,-1))[0]
            lp=log_w.copy()
            for k in range(K):
                diff=tmp_s[obs_c]-mu_scaled[k,obs_c]; var=cov_diag[k,obs_c]+1e-6
                lp[k]+=-0.5*np.sum(diff**2/var+np.log(2*np.pi*var))
            lp-=lp.max(); probs=np.exp(lp); probs/=probs.sum()
        X[i,miss_c]=probs@mu_orig[:,miss_c]

    df_out=df_in.copy(); df_out[feat_cols]=X
    return df_out

print('✅ gmm_impute() defined')

In [ ]:
print(f'Validation missing: {df_val_miss[feat_cols_clean].isnull().mean().mean()*100:.1f}%')
print('Running GMM imputation on validation set ...')
t0 = time.time()
df_val_imputed = gmm_impute(df_val_miss, gmm, scaler, feat_cols_clean)

# Fix structural zeros — non-business users always 0
non_biz = df_val_miss['user_type'].isin(['salaried_private','salaried_govt','self_employed'])
for col in STRUCTURAL_ZERO_COLS:
    if col in feat_cols_clean: df_val_imputed.loc[non_biz,col]=0.0

print(f'✅ Done in {time.time()-t0:.0f}s | Remaining NaNs: {df_val_imputed[feat_cols_clean].isnull().sum().sum()}')

In [ ]:
print('Running Imputation QA (vs ground truth) ...')
errors={}
for col in feat_cols_clean:
    was_miss = df_val_miss[col].isnull()
    if was_miss.sum()<5: continue
    gt = df_val_gt.loc[was_miss,col].values
    ip = df_val_imputed.loc[was_miss,col].values
    rng_col=float(gt.max()-gt.min())+1e-6
    errors[col]={'n':int(was_miss.sum()),
                 'MAE':float(mean_absolute_error(gt,ip)),
                 'NMAE':float(mean_absolute_error(gt,ip)/rng_col),
                 'R2':float(r2_score(gt,ip))}

err_df=pd.DataFrame(errors).T.sort_values('NMAE')
err_df.to_csv('outputs/imputation_qa.csv')

print(f'\n  QA Summary across {len(err_df)} features:')
print(f'  Avg NMAE  : {err_df["NMAE"].mean()*100:.2f}%  (lower=better)')
print(f'  Avg R²    : {err_df["R2"].mean():.3f}  (higher=better)')
print(f'  Best R²   : {err_df["R2"].max():.3f}  ← {err_df["R2"].idxmax()}')
print(f'  Worst R²  : {err_df["R2"].min():.3f}  ← {err_df["R2"].idxmin()}')
print(f'  NMAE<10%  : {(err_df["NMAE"]<0.10).sum()} features')
print(f'  NMAE>25%  : {(err_df["NMAE"]>0.25).sum()} features')

# Plot NMAE
fig,axes=plt.subplots(1,2,figsize=(16,5))
top30=err_df['NMAE'].sort_values().head(30)
top30.plot(kind='barh',ax=axes[0],color=['#2ecc71' if v<0.1 else('#f39c12' if v<0.2 else '#e74c3c') for v in top30])
axes[0].axvline(0.1,color='green',ls='--',lw=1.5,label='10% threshold')
axes[0].set_title('Imputation Quality (NMAE)',fontweight='bold'); axes[0].legend()

best_f=err_df['R2'].idxmax(); rows_b=df_val_miss[best_f].isnull()
tv=df_val_gt.loc[rows_b,best_f]; iv=df_val_imputed.loc[rows_b,best_f]
axes[1].scatter(tv,iv,alpha=0.4,color='steelblue',s=15)
lims=[min(tv.min(),iv.min()),max(tv.max(),iv.max())]
axes[1].plot(lims,lims,'r--',lw=2,label='Perfect')
axes[1].set_title(f'True vs Imputed: {best_f}\nR²={err_df.loc[best_f,"R2"]:.3f}',fontweight='bold')
axes[1].set_xlabel('True'); axes[1].set_ylabel('Imputed'); axes[1].legend()
plt.suptitle('GMM Imputation Validation',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/imputation_qa.png',dpi=150,bbox_inches='tight'); plt.show()

## 🤖 Step 6 — Train XGBoost Credit Scoring Model

In [ ]:
le = LabelEncoder()
le.fit(df_train['user_type'])

def make_X(df, feat_cols, label_enc):
    X=df[feat_cols].copy()
    X['user_type_enc']=label_enc.transform(df['user_type'])
    return X

X_train_full = make_X(df_train,       feat_cols_clean, le)
y_train_full = df_train['credit_score'].values
X_val_full   = make_X(df_val_imputed, feat_cols_clean, le)
y_val_full   = df_val_imputed['credit_score'].values

# Combine train + imputed-val
X_all = pd.concat([X_train_full,X_val_full],ignore_index=True)
y_all = np.concatenate([y_train_full,y_val_full])

X_fit,X_hold,y_fit,y_hold = train_test_split(X_all,y_all,test_size=0.10,random_state=42)
print(f'Train : {X_fit.shape}  |  Hold-out : {X_hold.shape}')

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=5000, learning_rate=0.03, max_depth=7,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
    reg_alpha=0.5, reg_lambda=2.0, random_state=42,
    eval_metric='rmse', early_stopping_rounds=40,
    tree_method='hist', verbosity=0,
)
print('Training XGBoost ...')
t0=time.time()
model.fit(X_fit,y_fit,eval_set=[(X_hold,y_hold)],verbose=200)
print(f'\n✅ Best iteration: {model.best_iteration}  ({time.time()-t0:.0f}s)')

## 📈 Step 7 — Evaluate + Bias Audit

In [ ]:
y_pred = np.clip(model.predict(X_hold),300,900)
mae    = mean_absolute_error(y_hold,y_pred)
rmse   = float(np.sqrt(mean_squared_error(y_hold,y_pred)))
r2     = r2_score(y_hold,y_pred)

tb=to_band(y_hold); pb=to_band(y_pred)
band_acc=float((tb==pb).mean())

try:
    auc=roc_auc_score((tb=='A').astype(int),np.clip((y_pred-300)/600,0,1))
except: auc=float('nan')

print('='*52)
print('MODEL EVALUATION (Hold-out 10%)')
print('='*52)
print(f'  MAE           : {mae:.2f} score points')
print(f'  RMSE          : {rmse:.2f}')
print(f'  R²            : {r2:.4f}')
print(f'  Band Accuracy : {band_acc*100:.1f}%')
print(f'  AUC (Band A)  : {auc:.3f}')
print('='*52)

fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].scatter(y_hold,y_pred,alpha=0.2,s=4,color='steelblue')
axes[0].plot([300,900],[300,900],'r--',lw=2,label='Perfect')
axes[0].set_title(f'True vs Predicted\nR²={r2:.4f} MAE={mae:.1f}',fontweight='bold')
axes[0].set_xlabel('True'); axes[0].set_ylabel('Predicted'); axes[0].legend()

resid=y_pred-y_hold
axes[1].hist(resid,bins=60,color='coral',edgecolor='white',lw=0.3)
axes[1].axvline(0,color='black',lw=2)
axes[1].axvline(resid.mean(),color='red',ls='--',label=f'Mean={resid.mean():.1f}')
axes[1].set_title('Residual Distribution',fontweight='bold'); axes[1].legend()

cm=confusion_matrix(tb,pb,labels=['A','B','C','D'])
sns.heatmap(cm,annot=True,fmt='d',xticklabels=['A','B','C','D'],yticklabels=['A','B','C','D'],
            cmap='Blues',ax=axes[2],cbar=False)
axes[2].set_title(f'Risk Band Confusion\nAccuracy={band_acc*100:.1f}%',fontweight='bold')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
plt.suptitle('XGBoost Evaluation',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/model_evaluation.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
print('── Bias Audit (MAE by user type) ──────────────────────')
# Map hold-out back to user types via df_train
n_hold = len(y_hold)
ut_series = pd.concat([df_train['user_type'],df_val_imputed['user_type']],ignore_index=True)
_,hold_idx=train_test_split(range(len(X_all)),test_size=0.10,random_state=42)
hold_types=ut_series.iloc[hold_idx].values

bias_rows=[]
for ut in np.unique(hold_types):
    mask_ut=(hold_types==ut)
    if mask_ut.sum()<10: continue
    mae_ut = mean_absolute_error(y_hold[mask_ut],y_pred[mask_ut])
    bias_rows.append({'user_type':ut,'n':mask_ut.sum(),'MAE':round(mae_ut,2)})
    print(f'  {ut:22s}  MAE={mae_ut:.1f}  n={mask_ut.sum()}')

bias_df=pd.DataFrame(bias_rows)
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(bias_df['user_type'],bias_df['MAE'],color='steelblue',edgecolor='white')
ax.axhline(mae,color='red',ls='--',label=f'Overall MAE={mae:.1f}')
ax.set_title('Bias Audit: MAE by User Type',fontweight='bold')
ax.set_xlabel('User Type'); ax.set_ylabel('MAE (score points)')
ax.tick_params(axis='x',rotation=20); ax.legend()
plt.tight_layout(); plt.savefig('outputs/bias_audit.png',dpi=150,bbox_inches='tight'); plt.show()

In [ ]:
try:
    import shap
    print('Computing SHAP values (500 sample) ...')
    explainer = shap.TreeExplainer(model)
    X_shap    = X_hold.sample(500, random_state=42) if hasattr(X_hold,'sample') else pd.DataFrame(X_hold).sample(500,random_state=42)
    shap_vals = explainer.shap_values(X_shap)
    plt.figure(figsize=(10,8))
    shap.summary_plot(shap_vals, X_shap, max_display=20, show=False)
    plt.title('SHAP Feature Importance',fontweight='bold',fontsize=13)
    plt.tight_layout(); plt.savefig('outputs/shap_summary.png',dpi=150,bbox_inches='tight'); plt.show()
except ImportError:
    print('shap not installed — skipping SHAP plot')

## 🎯 Step 8 — Predict on Test Set

In [ ]:
print('Imputing test set missing values ...')
# Align columns
for col in feat_cols_clean:
    if col not in df_test_nolabel.columns: df_test_nolabel[col]=np.nan
if 'user_type' not in df_test_nolabel.columns: df_test_nolabel['user_type']='salaried_private'

t0=time.time()
df_test_imputed = gmm_impute(df_test_nolabel, gmm, scaler, feat_cols_clean)
non_biz_t = df_test_nolabel['user_type'].isin(['salaried_private','salaried_govt','self_employed'])
for col in STRUCTURAL_ZERO_COLS:
    if col in feat_cols_clean: df_test_imputed.loc[non_biz_t,col]=0.0
print(f'✅ Done in {time.time()-t0:.0f}s')

X_test = make_X(df_test_imputed, feat_cols_clean, le)
raw    = model.predict(X_test)
scores = np.clip(np.round(raw).astype(int),300,900)
bands  = to_band(scores)

decisions = pd.Series(bands).map({
    'A':'AUTO_APPROVE','B':'APPROVE_LOWER_LIMIT',
    'C':'MANUAL_REVIEW','D':'REJECT'}).values

results = df_test_imputed[['user_id','user_type']].copy()
results['predicted_score'] = scores
results['risk_band']       = bands
results['loan_decision']   = decisions
results.to_csv('outputs/predictions_output.csv',index=False)

print(f'\nTest Results ({len(results):,} users):')
print(f'  Score range : {scores.min()} – {scores.max()}')
print(f'  Mean score  : {scores.mean():.0f}')
print(f'  Bands:\n{pd.Series(bands).value_counts().sort_index().to_string()}')
print(f'  Decisions:\n{results["loan_decision"].value_counts().to_string()}')

fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].hist(scores,bins=40,color='steelblue',edgecolor='white',lw=0.4)
for thr,lbl,col in [(750,'A/B','#2ecc71'),(650,'B/C','#f39c12'),(550,'C/D','#e74c3c')]:
    axes[0].axvline(thr,color=col,ls='--',lw=2,label=lbl)
axes[0].set_title('Predicted Score Distribution (Test)',fontweight='bold'); axes[0].legend(fontsize=8)

bc2=pd.Series(bands).value_counts().sort_index()
pc={'A':'#2ecc71','B':'#f39c12','C':'#e67e22','D':'#e74c3c'}
axes[1].pie(bc2.values,labels=[f'Band {b}\n({c:,})' for b,c in bc2.items()],
            colors=[pc[b] for b in bc2.index],autopct='%1.1f%%',startangle=90,textprops={'fontsize':9})
axes[1].set_title('Risk Band Distribution (Test)',fontweight='bold')

results.boxplot(column='predicted_score',by='user_type',ax=axes[2])
axes[2].set_title('Score by User Type',fontweight='bold')
axes[2].set_xlabel('User Type'); axes[2].set_ylabel('Score')
axes[2].tick_params(axis='x',rotation=30); plt.sca(axes[2]); plt.title('Score by User Type')
fig.suptitle('Final Test Set Predictions',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('outputs/final_predictions.png',dpi=150,bbox_inches='tight'); plt.show()

## 🚀 Step 9 — Save Models + Real-Time New User Scoring
Simulates a **new user entering the system** from the production database — their raw transaction history is passed in, features are engineered, missing values are imputed, and a credit score is returned.

In [ ]:
joblib.dump(scaler,         'models/scaler.pkl')
joblib.dump(gmm,            'models/gmm_model.pkl')
joblib.dump(model,          'models/xgb_model.pkl')
joblib.dump(le,             'models/label_encoder.pkl')
joblib.dump(feat_cols_clean,'models/feature_cols.pkl')
joblib.dump(STRUCTURAL_ZERO_COLS,'models/structural_zero_cols.pkl')
print('✅ All models saved to models/')

In [ ]:
# Load once at module level — production safe (no repeated disk reads)
_scaler    = joblib.load('models/scaler.pkl')
_gmm       = joblib.load('models/gmm_model.pkl')
_model     = joblib.load('models/xgb_model.pkl')
_le        = joblib.load('models/label_encoder.pkl')
_feat_cols = joblib.load('models/feature_cols.pkl')
_zero_cols = joblib.load('models/structural_zero_cols.pkl')

def score_new_user(raw_txn_df, user_profile):
    """
    Score a brand-new user from their raw transaction DataFrame.

    raw_txn_df   : DataFrame with [date, month, category, direction, amount]
    user_profile : dict with keys: user_id, user_type, age, n_months,
                   monthly_income, has_emi, is_business,
                   employer_tenure, account_vintage, kyc_score, co_applicant
    """
    # 1. Feature engineering from raw transactions
    feat = engineer_features(user_profile.get('user_id',0), raw_txn_df, user_profile)
    if feat is None: return {'error':'No transactions found'}
    feat['user_type'] = user_profile['user_type']

    # 2. GMM soft-cluster imputation for any missing features
    row = np.array([feat.get(c, np.nan) for c in _feat_cols], dtype=float)
    mask = np.isnan(row); miss_idx=np.where(mask)[0]; obs_idx=np.where(~mask)[0]
    imputed_cols=[_feat_cols[i] for i in miss_idx]

    if len(miss_idx)>0:
        mu_s=_gmm.means_; mu_o=_scaler.inverse_transform(_gmm.means_)
        cv=_gmm.covariances_; lw=np.log(_gmm.weights_+1e-10); K=_gmm.n_components
        if len(obs_idx)==0:
            probs=np.exp(lw-lw.max()); probs/=probs.sum()
        else:
            tmp=row.copy(); tmp[miss_idx]=mu_o[:,miss_idx].mean(axis=0)
            tmp_s=_scaler.transform(tmp.reshape(1,-1))[0]
            lp=lw.copy()
            for k in range(K):
                d=tmp_s[obs_idx]-mu_s[k,obs_idx]; v=cv[k,obs_idx]+1e-6
                lp[k]+=-0.5*np.sum(d**2/v+np.log(2*np.pi*v))
            lp-=lp.max(); probs=np.exp(lp); probs/=probs.sum()
        row[miss_idx]=probs@mu_o[:,miss_idx]
        # Override structural zeros
        for col in _zero_cols:
            if col in _feat_cols and user_profile['user_type'] not in ['shopkeeper','businessman']:
                row[_feat_cols.index(col)]=0.0

    feat_imputed={_feat_cols[i]:float(row[i]) for i in range(len(_feat_cols))}

    # 3. XGBoost predict
    X = pd.DataFrame([feat_imputed])
    X['user_type_enc']=_le.transform([user_profile['user_type']])
    raw_score=float(_model.predict(X[_feat_cols+['user_type_enc']])[0])
    score=int(np.clip(round(raw_score),300,900))
    band ='A' if score>=750 else ('B' if score>=650 else ('C' if score>=550 else 'D'))
    dec  ={'A':'AUTO_APPROVE','B':'APPROVE_LOWER_LIMIT','C':'MANUAL_REVIEW','D':'REJECT'}[band]

    # 4. Cluster membership
    feat_arr=np.array([feat_imputed.get(c,0.0) for c in _feat_cols])
    feat_sc =_scaler.transform(feat_arr.reshape(1,-1))
    cluster_probs=_gmm.predict_proba(feat_sc)[0]
    cluster_dict={f'C{i}':round(float(p),3) for i,p in enumerate(cluster_probs)}

    # 5. Score dimension breakdown
    f=feat_imputed
    def c(x): return float(np.clip(x,0,1))
    dims={
        'Income stability':    round(c(f.get('salary_day_consistency',0)*0.4+f.get('net_savings_rate',0)*0.3+(1-f.get('income_variability_cv',0.5)/2)*0.3)*100,1),
        'Balance behaviour':   round(c(1-f.get('balance_utilisation_ratio',0.5))*100,1),
        'Spending discipline': round(c(1-f.get('spend_to_income_ratio',0.7))*100,1),
        'EMI repayment':       round(c(f.get('emi_paid_ontime_ratio',0.5))*100,1),
        'Bill payments':       round(c(f.get('utility_payment_consistency',0.5))*100,1),
        'Savings':             round(c(f.get('net_savings_rate',0)/0.55+0.18)*100,1),
        'BNPL behaviour':      round(c(f.get('bnpl_repayment_ratio',0.8))*100,1),
        'Digital engagement':  round(c(f.get('app_sessions_monthly',0)/60)*100,1),
    }
    sorted_dims=sorted(dims.items(),key=lambda x:x[1],reverse=True)

    return {
        'user_id':            user_profile.get('user_id','NEW'),
        'user_type':          user_profile['user_type'],
        'credit_score':       score,
        'risk_band':          band,
        'loan_decision':      dec,
        'top_positive_factors':{k:v for k,v in sorted_dims[:3]},
        'top_negative_factors':{k:v for k,v in sorted_dims[-3:]},
        'cluster_membership': cluster_dict,
        'primary_cluster':    int(np.argmax(cluster_probs)),
        'imputed_features':   imputed_cols,
        'history_months':     user_profile.get('n_months',12),
    }

print('✅ score_new_user() defined and models loaded')

In [ ]:
# ── DEMO: Score 3 new users from their raw transaction history ────
demo_cases = [
    ('salaried_private','A','High-earning salaried — expect GOOD score'),
    ('shopkeeper',      'C','Struggling shopkeeper — expect MEDIUM score'),
    ('businessman',     'D','High-risk businessman — expect LOW score'),
]

for utype,band,desc in demo_cases:
    print(f'\n{"─"*60}')
    print(f'Case: {desc}')
    rng_d   = default_rng(hash((utype,band,99))&0xFFFFFFFF)
    profile = build_profile(9999,utype,band,rng_d)
    txns    = generate_user_transactions(profile,rng_d)
    txn_df  = pd.DataFrame(txns)
    # Drop 25% of months to simulate partial history
    drop_m  = rng_d.choice(range(1,profile['n_months']+1),size=max(1,profile['n_months']//4),replace=False)
    txn_df  = txn_df[~txn_df['month'].isin(drop_m)]

    out = score_new_user(txn_df, profile)

    print(f"  Credit Score  : {out['credit_score']} / 900")
    print(f"  Risk Band     : {out['risk_band']}")
    print(f"  Decision      : {out['loan_decision']}")
    print(f"  History used  : {out['history_months']} months  |  Imputed: {len(out['imputed_features'])} features")
    print(f"  +ve Factors   : {out['top_positive_factors']}")
    print(f"  -ve Factors   : {out['top_negative_factors']}")
    print(f"  Top cluster   : C{out['primary_cluster']}  ({max(out['cluster_membership'].values())*100:.0f}% membership)")

## ✅ Pipeline Summary

| Step | Task | Output |
|------|------|--------|
| 1 | Raw transaction generation (1L users, 12–24 months) | `data/raw_transactions.csv` |
| 2 | Feature engineering (62 features from raw txns) | computed in-memory |
| 3 | EDA + Train/Val/Test splits | `data/training.csv`, `validation.csv`, `test.csv` |
| 4 | GMM clustering (BIC-optimal n) | behavioral cluster profiles |
| 5 | GMM imputation + QA vs ground truth | `outputs/imputation_qa.csv` |
| 6 | XGBoost training (train + imputed-val) | credit score model |
| 7 | Evaluation: MAE, RMSE, R², AUC, Bias Audit | `outputs/model_evaluation.png` |
| 8 | Test predictions | `outputs/predictions_output.csv` |
| 9 | Real-time new user scoring | `score_new_user(txn_df, profile)` |

**Models saved:** `models/gmm_model.pkl` · `models/xgb_model.pkl` · `models/scaler.pkl`

**Features:** 62 behavioral features across 10 dimensions including **BNPL** (new), salary day consistency, inflow/outflow ratio, EMI bounce count, GST payments (business only).